# Simulated Annealing QUBO 小规模验证

这本 notebook 验证纯本地 `SimulatedAnnealingQuboSolver` 的协议、候选能量与固定 seed 重复性。算法逻辑和 deterministic fixtures 都来自仓库脚本；notebook 只导入、配置、调用、展示和执行简单断言。

> Simulated Annealing 是启发式算法。正常完成应返回 `feasible`，即使命中精确最优解也不能声明 `optimal`。

## 1. 环境与导入

在 VS Code/Jupyter 中选择项目 `.venv` 对应的 Python kernel。下面只负责定位仓库根目录和导入现有组件。

In [ ]:
import json
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_qubo, validate_qubo_result
from lib.solvers.qubo import ExactQuboSolver, SimulatedAnnealingQuboSolver
from problem.benchmarks import build_annealing_validation_suite
from tests.oracles import enumerate_qubo, evaluate_qubo_energy, public_json_number

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2. 加载确定性 fixtures

`build_annealing_validation_suite()` 的实例由 Python 脚本定义。这里选择一个五变量稀疏问题作为主验证实例，并选择一个二变量问题验证 custom schedule。

In [ ]:
suite = build_annealing_validation_suite()
problem = suite["sparse_random"]
custom_problem = suite["custom_schedule"]

validate_qubo(problem)
validate_qubo(custom_problem)

print("Available fixtures:", sorted(suite))
print("Main problem:", problem["problem_id"])
print("Variables:", problem["variable_names"])
print("Terms:", len(problem["terms"]))

## 3. 独立 oracle 与 Exact 对照

`tests.oracles` 使用独立的 `Fraction` 穷举实现，不调用 production energy helper 或任何 solver。`ExactQuboSolver` 则验证同一问题在仓库 solver 协议下的精确结果。

In [ ]:
oracle_rows = enumerate_qubo(problem)
oracle_sample = oracle_rows[0]["sample"]
oracle_energy = public_json_number(oracle_rows[0]["energy_exact"])

exact_result = ExactQuboSolver().solve(problem)
validate_qubo_result(problem, exact_result)

assert exact_result["status"] == "optimal"
assert exact_result["best_sample"] == oracle_sample
assert exact_result["best_energy"] == oracle_energy

print("Oracle optimum:", oracle_sample, "energy =", oracle_energy)
print("Enumerated assignments:", len(oracle_rows))

## 4. 运行 Simulated Annealing

固定预算和 seed 后运行 SA。独立 oracle 重新计算返回 sample 的能量；gap 只衡量这次运行与精确最优值的距离，不构成最优性证明。

In [ ]:
sa_config = {
    "num_reads": 8,
    "sweeps": 50,
    "seed": 1729,
    "update_order": "random",
}

sa_solver = SimulatedAnnealingQuboSolver()
sa_result = sa_solver.solve(problem, sa_config)
validate_qubo_result(problem, sa_result)

observed_energy = public_json_number(
    evaluate_qubo_energy(problem, sa_result["best_sample"])
)
optimality_gap = sa_result["best_energy"] - exact_result["best_energy"]

assert sa_result["status"] == "feasible"
assert sa_result["status"] != "optimal"
assert sa_result["best_energy"] == observed_energy
assert optimality_gap >= 0

print(json.dumps(sa_result, indent=2, ensure_ascii=False))
print("Exact gap:", optimality_gap)

## 5. 固定 seed 重复性

相同问题与配置应重复算法拥有的结果字段。`runtime_seconds` 来自墙钟计时，所以不参与相等断言。

In [ ]:
repeated_result = SimulatedAnnealingQuboSolver().solve(problem, sa_config)
validate_qubo_result(problem, repeated_result)

assert repeated_result["status"] == sa_result["status"]
assert repeated_result["best_sample"] == sa_result["best_sample"]
assert repeated_result["best_energy"] == sa_result["best_energy"]
assert repeated_result["termination_reason"] == sa_result["termination_reason"]
assert repeated_result["metrics"] == sa_result["metrics"]
assert repeated_result["metadata"] == sa_result["metadata"]

print("Fixed-seed algorithm fields repeat exactly.")

## 6. Custom inverse-temperature schedule

Custom schedule 是非负、单调不减且长度等于 `sweeps` 的显式 beta 序列。这里同时给出初态和顺序更新，使实验配置完全可见。

In [ ]:
custom_config = {
    "num_reads": 2,
    "sweeps": 4,
    "beta_schedule_type": "custom",
    "beta_schedule": [0.0, 0.25, 1.0, 4.0],
    "initial_samples": [[0, 0], [1, 1]],
    "update_order": "sequential",
    "seed": 23,
}

custom_result = SimulatedAnnealingQuboSolver().solve(
    custom_problem,
    custom_config,
)
validate_qubo_result(custom_problem, custom_result)

custom_observed_energy = public_json_number(
    evaluate_qubo_energy(custom_problem, custom_result["best_sample"])
)

assert custom_result["status"] == "feasible"
assert custom_result["status"] != "optimal"
assert custom_result["best_energy"] == custom_observed_energy
assert custom_result["metadata"]["beta_schedule_type"] == "custom"
assert custom_result["metadata"]["beta_start"] == 0.0
assert custom_result["metadata"]["beta_end"] == 4.0

print(json.dumps(custom_result, indent=2, ensure_ascii=False))

## 结论与边界

- SA 直接遵循 `qubo.v1 -> qubo-result.v1`，返回 sample 的能量通过独立 oracle 和公共契约双重检查；
- 固定 seed 可以复现算法字段，但不要求墙钟 runtime 相同；
- custom schedule 由调用方显式配置，notebook 不实现 schedule 或 Metropolis 逻辑；
- SA 的正确状态是 `feasible` 或 `timeout`，永不因为命中最优值而返回 `optimal`；
- timeout 通过单元测试中的可控 deadline seam 验证；这里不使用依赖真实运行时间的脆弱演示。